# GlucoSense Shredder v1 — Deep Learning
## 1D CNN + LSTM on Real Antenna S11 Sweeps

**Same 81 sweeps as Annihilator v1.** Deep learning path.

**Fixes applied:**
- Gradient clipping on LSTM (clipnorm=1.0) — prevents explosion on 1001 timesteps
- Mixed precision enabled for RTX 4080 (2-3x speedup on Ampere)
- GPU auto-detection (CUDA for Alienware, Metal for M2)
- Noise augmentation: sigma=0.05 dB (calibrated RF noise floor), train only

**RTX 4080 install (Alienware):**
```
pip install tensorflow[and-cuda]   # TF 2.13+ with CUDA auto-config
# OR: pip install tensorflow-gpu   # older TF
```
**M2 install (MacBook):**
```
pip install tensorflow-macos tensorflow-metal
```

In [ ]:
# GPU/Platform Detection + TF Setup
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'tensorflow', 'scikit-learn', 'pandas', 'numpy', 'matplotlib', 'catboost'])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings, os, pickle
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, mixed_precision
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from catboost import CatBoostRegressor

SEED = 42
DATA_DIR = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else '.'
np.random.seed(SEED)
tf.random.set_seed(SEED)

# GPU detection
gpus = tf.config.list_physical_devices('GPU')
platform = 'CPU'
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        platform = f'GPU ({len(gpus)}x)'
        # Mixed precision: 2-3x faster on RTX 4080 (Ampere) and Apple M-series
        mixed_precision.set_global_policy('mixed_float16')
        print(f'Mixed precision enabled (float16) — RTX 4080 / M-series optimized')
    except Exception as e:
        print(f'GPU setup warning: {e}')
else:
    # CPU: disable mixed precision (float32 faster on CPU)
    mixed_precision.set_global_policy('float32')

print(f'Platform: {platform}')
print(f'TensorFlow: {tf.__version__}')
print(f'Devices: {[d.device_type for d in tf.config.list_physical_devices()]}')

In [ ]:
# Load + Restructure (same cleaning as Annihilator)
raw = pd.read_csv(os.path.join(DATA_DIR, 'NEW_BGL_DATASET.csv'))
raw.columns = ['Frequency', 'S11_dB', 'BGL']
raw = raw.apply(pd.to_numeric, errors='coerce').dropna()
raw['Frequency'] = raw['Frequency'].round(4)  # collapse float-precision duplicates
idx_95 = raw[raw['BGL'] == 95].index
raw    = raw.drop(idx_95[1001:])

sweep_df  = raw.pivot_table(index='BGL', columns='Frequency', values='S11_dB', aggfunc='first').sort_index()
BGL_vals  = sweep_df.index.values.astype(float)
FREQ_pts  = sweep_df.columns.values.astype(float)
X_sweep   = sweep_df.values.astype(np.float32)
y_bgl     = BGL_vals.astype(np.float32)
N         = len(X_sweep)

# 4-class labels
y_class = np.array(pd.cut(y_bgl, bins=[0,140,200,300,500], labels=range(4))).astype(int)

print(f'Sweeps: {X_sweep.shape}  BGL: {int(y_bgl.min())}-{int(y_bgl.max())} mg/dL  N={N}')

In [ ]:
# Noise Augmentation
# sigma=0.05 dB = conservative VNA/power-meter noise floor (real RF measurement)
# Applied ONLY to training set inside LOO fold — never to test sample
NOISE_STD  = 0.05   # dB
N_AUGMENTS = 15     # 80 train x 15 = 1200 train samples per fold

def augment_sweeps(X, y, noise_std=NOISE_STD, n_aug=N_AUGMENTS, seed=42):
    rng  = np.random.RandomState(seed)
    X_out, y_out = [X.copy()], [y.copy()]
    for _ in range(n_aug):
        X_out.append(X + rng.normal(0, noise_std, X.shape).astype(np.float32))
        y_out.append(y.copy())
    return np.concatenate(X_out), np.concatenate(y_out)

Xd, yd = augment_sweeps(X_sweep[:3], y_bgl[:3])
print(f'Demo: 3 x (1+{N_AUGMENTS}) = {len(Xd)} augmented training samples')
print(f'Noise: sigma={NOISE_STD} dB per frequency point')

In [ ]:
# Model Architectures

def build_1d_cnn(input_len=1001, dropout=0.4, l2=1e-4):
    reg = keras.regularizers.l2(l2)
    inp = keras.Input(shape=(input_len, 1))
    x   = layers.Conv1D(32, 15, padding='same', activation='relu', kernel_regularizer=reg)(inp)
    x   = layers.BatchNormalization()(x)
    x   = layers.MaxPooling1D(4)(x)        # 1001 -> 250
    x   = layers.Dropout(dropout)(x)
    x   = layers.Conv1D(64, 7, padding='same', activation='relu', kernel_regularizer=reg)(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.MaxPooling1D(4)(x)        # 250 -> 62
    x   = layers.Dropout(dropout)(x)
    x   = layers.Conv1D(128, 3, padding='same', activation='relu', kernel_regularizer=reg)(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.GlobalAveragePooling1D()(x)
    x   = layers.Dense(64, activation='relu', kernel_regularizer=reg)(x)
    x   = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation='linear', dtype='float32')(x)  # float32 output for mixed precision
    m   = keras.Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mae')
    return m

def build_lstm(input_len=1001, units=64, dropout=0.4):
    # FIX: clipnorm=1.0 prevents gradient explosion on 1001 timesteps
    # FIX: recurrent_dropout added to prevent RNN-specific overfit
    inp = keras.Input(shape=(input_len, 1))
    x   = layers.LSTM(units, return_sequences=True,
                      dropout=dropout, recurrent_dropout=0.2)(inp)
    x   = layers.LSTM(units//2, return_sequences=False,
                      dropout=dropout, recurrent_dropout=0.2)(x)
    x   = layers.Dense(32, activation='relu')(x)
    x   = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation='linear', dtype='float32')(x)  # float32 for mixed precision
    m   = keras.Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(1e-3, clipnorm=1.0), loss='mae')  # FIX: clipnorm
    return m

# Test build
cnn_test = build_1d_cnn()
cnn_test.summary()

In [ ]:
# LOO-CV: 1D CNN Regression
print(f'Running LOO-CV: 1D CNN (81 folds)...')
loo = LeaveOneOut()
y_pred_cnn = np.zeros(N, dtype=np.float32)

for i, (tr, te) in enumerate(loo.split(X_sweep)):
    Xtr_aug, ytr_aug = augment_sweeps(X_sweep[tr], y_bgl[tr], seed=SEED+i)
    mu, std = Xtr_aug.mean(), Xtr_aug.std() + 1e-8
    Xtr_n   = ((Xtr_aug - mu) / std).reshape(-1, 1001, 1)
    Xte_n   = ((X_sweep[te] - mu) / std).reshape(-1, 1001, 1)

    model = build_1d_cnn(dropout=0.4)
    es    = callbacks.EarlyStopping(monitor='val_loss', patience=15,
                                     restore_best_weights=True, verbose=0)
    model.fit(Xtr_n, ytr_aug, epochs=120, batch_size=32,
              validation_split=0.10, callbacks=[es], verbose=0)
    y_pred_cnn[te] = float(model.predict(Xte_n, verbose=0).squeeze())

    if (i+1) % 15 == 0:
        mae_so_far = mean_absolute_error(y_bgl[:i+1], y_pred_cnn[:i+1])
        print(f'  Fold {i+1:2d}/81  running MAE: {mae_so_far:.2f} mg/dL')
    keras.backend.clear_session()

cnn_mae  = mean_absolute_error(y_bgl, y_pred_cnn)
cnn_r2   = r2_score(y_bgl, y_pred_cnn)
cnn_mape = np.mean(np.abs((y_bgl - y_pred_cnn) / y_bgl)) * 100
cnn_w15  = np.mean(np.abs(y_bgl - y_pred_cnn) <= np.maximum(15.0, 0.15*y_bgl)) * 100
print(f'\n1D CNN: MAE={cnn_mae:.2f}  R2={cnn_r2:.4f}  MAPE={cnn_mape:.1f}%  Within15={cnn_w15:.1f}%')
print('Note: Adjusted R2 NOT reported for neural nets (p = millions of weights)')

In [ ]:
# LOO-CV: LSTM Regression
print('Running LOO-CV: LSTM (81 folds)...')
y_pred_lstm = np.zeros(N, dtype=np.float32)

for i, (tr, te) in enumerate(loo.split(X_sweep)):
    Xtr_aug, ytr_aug = augment_sweeps(X_sweep[tr], y_bgl[tr], n_aug=10, seed=SEED+i)
    mu, std = Xtr_aug.mean(), Xtr_aug.std() + 1e-8
    Xtr_n   = ((Xtr_aug - mu) / std).reshape(-1, 1001, 1)
    Xte_n   = ((X_sweep[te] - mu) / std).reshape(-1, 1001, 1)

    model = build_lstm(units=32, dropout=0.4)  # small for CPU; increase on GPU
    es    = callbacks.EarlyStopping(monitor='val_loss', patience=10,
                                     restore_best_weights=True, verbose=0)
    model.fit(Xtr_n, ytr_aug, epochs=80, batch_size=32,
              validation_split=0.10, callbacks=[es], verbose=0)
    y_pred_lstm[te] = float(model.predict(Xte_n, verbose=0).squeeze())
    if (i+1) % 15 == 0:
        print(f'  Fold {i+1:2d}/81')
    keras.backend.clear_session()

lstm_mae  = mean_absolute_error(y_bgl, y_pred_lstm)
lstm_r2   = r2_score(y_bgl, y_pred_lstm)
lstm_mape = np.mean(np.abs((y_bgl - y_pred_lstm) / y_bgl)) * 100
lstm_w15  = np.mean(np.abs(y_bgl - y_pred_lstm) <= np.maximum(15.0, 0.15*y_bgl)) * 100
print(f'\nLSTM: MAE={lstm_mae:.2f}  R2={lstm_r2:.4f}  MAPE={lstm_mape:.1f}%  Within15={lstm_w15:.1f}%')

In [ ]:
# CatBoost LOO-CV + Ensemble

def extract_features_simple(sweeps, freqs):
    n   = len(sweeps)
    fi  = np.argmin(sweeps, axis=1)
    out = {}
    out['res_freq']  = freqs[fi]
    out['res_s11']   = sweeps[np.arange(n), fi]
    out['auc']       = -np.trapz(sweeps, freqs, axis=1)
    out['s11_std']   = sweeps.std(axis=1)
    out['s11_mean']  = sweeps.mean(axis=1)
    for lo, hi in [(1.0,2.0),(2.0,3.0),(3.0,4.0),(4.0,5.0)]:
        mask = (freqs >= lo) & (freqs < hi)
        out[f'mean_{int(lo)}{int(hi)}'] = sweeps[:, mask].mean(axis=1)
    for f_hz in [1.5, 2.5, 3.5, 4.5]:
        out[f's11_{f_hz:.1f}'] = sweeps[:, np.argmin(np.abs(freqs-f_hz))]
    return pd.DataFrame(out).values.astype(np.float32)

X_feat = extract_features_simple(X_sweep, FREQ_pts)
print(f'Features: {X_feat.shape}')

y_pred_cb = np.zeros(N)
for tr, te in loo.split(X_feat):
    sc = StandardScaler()
    cb = CatBoostRegressor(iterations=800, learning_rate=0.05, depth=5, verbose=0, random_state=SEED)
    cb.fit(sc.fit_transform(X_feat[tr]), y_bgl[tr])
    y_pred_cb[te] = cb.predict(sc.transform(X_feat[te]))

cb_mae = mean_absolute_error(y_bgl, y_pred_cb)
cb_r2  = r2_score(y_bgl, y_pred_cb)
print(f'CatBoost: MAE={cb_mae:.2f}  R2={cb_r2:.4f}')

# Ensemble: weighted by inverse MAE
maes_inv = 1.0 / np.array([cnn_mae, lstm_mae, cb_mae])
w        = maes_inv / maes_inv.sum()
y_ens    = w[0]*y_pred_cnn + w[1]*y_pred_lstm + w[2]*y_pred_cb
ens_mae  = mean_absolute_error(y_bgl, y_ens)
ens_r2   = r2_score(y_bgl, y_ens)
print(f'\nEnsemble weights (inv-MAE): CNN={w[0]:.3f}  LSTM={w[1]:.3f}  CatBoost={w[2]:.3f}')
print(f'Ensemble: MAE={ens_mae:.2f}  R2={ens_r2:.4f}')

In [ ]:
# Final plots + save

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle('GlucoSense Shredder v1 — DL Results (LOO-CV + noise augment)', fontweight='bold')

for ax, (nm, pred) in zip(axes, [('1D CNN', y_pred_cnn),('LSTM', y_pred_lstm),
                                   ('CatBoost', y_pred_cb),('Ensemble', y_ens)]):
    mae = mean_absolute_error(y_bgl, pred)
    r2  = r2_score(y_bgl, pred)
    ax.scatter(y_bgl, pred, s=40, alpha=0.8)
    ax.plot([70,490],[70,490],'k--',alpha=0.4)
    ax.set_title(f'{nm}\nMAE={mae:.2f}  R2={r2:.4f}', fontweight='bold')
    ax.set_xlabel('True BGL (mg/dL)'); ax.set_ylabel('Pred BGL (mg/dL)')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'shredder_results.png'), dpi=100, bbox_inches='tight')
plt.show()
print('Saved: shredder_results.png')

# Save final CNN on all data
Xfull_aug, yfull_aug = augment_sweeps(X_sweep, y_bgl, n_aug=N_AUGMENTS, seed=SEED)
mu_all = Xfull_aug.mean(); std_all = Xfull_aug.std() + 1e-8
Xfull_n = ((Xfull_aug - mu_all) / std_all).reshape(-1, 1001, 1)
final_cnn = build_1d_cnn()
final_cnn.fit(Xfull_n, yfull_aug, epochs=80, batch_size=32, verbose=0)
final_cnn.save(os.path.join(DATA_DIR, 'glucosense_cnn_final.keras'))
meta = {'mu': float(mu_all), 'std': float(std_all), 'freq_pts': FREQ_pts.tolist(),
        'noise_std': NOISE_STD, 'n_aug': N_AUGMENTS}
with open(os.path.join(DATA_DIR, 'glucosense_cnn_meta.pkl'), 'wb') as f: pickle.dump(meta, f)

pd.DataFrame({'BGL_true': y_bgl, 'pred_cnn': y_pred_cnn,
              'pred_lstm': y_pred_lstm, 'pred_cb': y_pred_cb, 'pred_ensemble': y_ens}
).to_csv(os.path.join(DATA_DIR, 'shredder_predictions.csv'), index=False)

print('\n=== SHREDDER v1 — FINAL SUMMARY ===')
for nm, pred in [('CatBoost (ML)', y_pred_cb),('1D CNN',y_pred_cnn),
                  ('LSTM', y_pred_lstm),('Ensemble', y_ens)]:
    mae  = mean_absolute_error(y_bgl, pred)
    r2   = r2_score(y_bgl, pred)
    mape = np.mean(np.abs((y_bgl-pred)/y_bgl))*100
    w15  = np.mean(np.abs(y_bgl-pred)<=np.maximum(15.0,0.15*y_bgl))*100
    print(f'{nm:<20} MAE={mae:6.2f}  R2={r2:.4f}  MAPE={mape:5.1f}%  Within15={w15:5.1f}%')